# Speaker Verification: ResNet-50 + ArcFace Loss (5s Audio)

## Why ArcFace?
| Approach | How it works | Limitation |
|:---------|:-------------|:-----------|
| Contrastive Loss | Pulls same pairs close, pushes different pairs apart | Only sees 2 speakers at a time |
| Triplet Loss | Ranks positive closer than negative | Only sees 3 speakers at a time |
| **ArcFace** | **Classifies ALL speakers simultaneously** with angular margin | **Sees all speakers every batch** |

**ArcFace** adds an angular penalty to the classification boundary:
$$L = -\log \frac{e^{s \cdot \cos(\theta_{y_i} + m)}}{e^{s \cdot \cos(\theta_{y_i} + m)} + \sum_{j \neq y_i} e^{s \cdot \cos(\theta_j)}}$$

Where `s=30` (scale), `m=0.5` (angular margin). The margin forces tight intra-class clustering.

### Training Strategy
- **Phase 1 (Training):** Train as a classification task (predict speaker ID)
- **Phase 2 (Evaluation):** Extract embeddings and measure cosine similarity on test pairs


In [ ]:
import os
import pandas as pd
import numpy as np

DATASET_ROOT   = "/kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset"
BASE_AUDIO_DIR = os.path.join(DATASET_ROOT, "train-clean-100", "train-clean-100")
TRAIN_CSV      = os.path.join(DATASET_ROOT, "train_pairs.csv")

TEST_ROOT      = "/kaggle/input/datasets/tahmidulislamomi/ml-project-testing-dataset"
TEST_AUDIO_DIR = os.path.join(TEST_ROOT, "test-clean")
TEST_CSV       = os.path.join(TEST_ROOT, "test_pairs.csv")

print("Train audio dir:", BASE_AUDIO_DIR)
print("Test  audio dir:", TEST_AUDIO_DIR)

## 2. Load Data & Build Speaker Mapping

ArcFace needs a **classification target** (integer class ID for each speaker). We build a mapping from speaker folder names to integer IDs.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

def to_train_abs(rel):
    return os.path.join(BASE_AUDIO_DIR, rel.replace("train-clean-100/", "", 1))
def to_test_abs(rel):
    return os.path.join(TEST_AUDIO_DIR, rel)

train_df["path1_abs"] = train_df["audio_path_1"].apply(to_train_abs)
train_df["path2_abs"] = train_df["audio_path_2"].apply(to_train_abs)
test_df["path1_abs"]  = test_df["audio_path_1"].apply(to_test_abs)
test_df["path2_abs"]  = test_df["audio_path_2"].apply(to_test_abs)

# Extract speaker IDs from paths
def get_speaker(rel_path):
    return rel_path.split("/")[1]  # "train-clean-100/311/..." → "311"

train_df["speaker1"] = train_df["audio_path_1"].apply(get_speaker)
train_df["speaker2"] = train_df["audio_path_2"].apply(get_speaker)

# Build unique speaker → integer class ID mapping
all_speakers = sorted(set(train_df["speaker1"].unique()) | set(train_df["speaker2"].unique()))
speaker_to_id = {spk: idx for idx, spk in enumerate(all_speakers)}
NUM_SPEAKERS = len(speaker_to_id)

print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")
print(f"Unique speakers: {NUM_SPEAKERS}")
print(f"Speaker map sample: {list(speaker_to_id.items())[:5]}")

## 3. Audio Transforms — 5s Window

In [ ]:
import torch
import torchaudio.transforms as T

TARGET_SR     = 16000
TARGET_LENGTH = TARGET_SR * 5   # 80000 samples (5 seconds)

def crop_or_pad(audio, is_train=True):
    length = len(audio)
    if length > TARGET_LENGTH:
        if is_train:
            start = np.random.randint(0, length - TARGET_LENGTH)
        else:
            start = (length - TARGET_LENGTH) // 2
        audio = audio[start:start + TARGET_LENGTH]
    elif length < TARGET_LENGTH:
        audio = np.pad(audio, (0, TARGET_LENGTH - length), mode='constant')
    return audio

mel_transform   = T.MelSpectrogram(sample_rate=16000, n_fft=400, hop_length=160, n_mels=80)
amplitude_to_db = T.AmplitudeToDB()
print("Transforms ready. Target duration: 5 seconds.")

## 4. Datasets

**ArcFaceDataset:** Returns `(spectrogram, speaker_class_id)` for classification training.

**SpeakerPairDataset:** Returns `(mel1, mel2, label)` for pair-based evaluation.

In [ ]:
import soundfile as sf
from torch.utils.data import Dataset, DataLoader

class ArcFaceDataset(Dataset):
    """Returns (mel_spectrogram, speaker_class_id) for classification training."""
    def __init__(self, dataframe, mel_transform, amplitude_to_db, speaker_to_id):
        self.mel_transform   = mel_transform
        self.amplitude_to_db = amplitude_to_db
        self.speaker_to_id   = speaker_to_id

        # Collect all unique (audio_path, speaker_id) pairs
        samples = set()
        for _, row in dataframe.iterrows():
            samples.add((row["path1_abs"], row["speaker1"]))
            samples.add((row["path2_abs"], row["speaker2"]))
        self.samples = list(samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, speaker = self.samples[idx]
        audio, sr = sf.read(path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        audio = crop_or_pad(audio, is_train=True)
        audio = torch.tensor(audio).float().unsqueeze(0)
        mel   = self.amplitude_to_db(self.mel_transform(audio))
        label = self.speaker_to_id[speaker]
        return mel, label


class SpeakerPairDataset(Dataset):
    """Returns (mel1, mel2, label) for pair-based evaluation."""
    def __init__(self, dataframe, mel_transform, amplitude_to_db):
        self.df              = dataframe
        self.mel_transform   = mel_transform
        self.amplitude_to_db = amplitude_to_db

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):
        audio, sr = sf.read(path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        audio = crop_or_pad(audio, is_train=False)
        audio = torch.tensor(audio).float().unsqueeze(0)
        return self.amplitude_to_db(self.mel_transform(audio))

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        mel1  = self.load_audio(row["path1_abs"])
        mel2  = self.load_audio(row["path2_abs"])
        label = torch.tensor(row["label"]).float()
        return mel1, mel2, label

In [ ]:
# ArcFace training loader
arcface_dataset = ArcFaceDataset(train_df, mel_transform, amplitude_to_db, speaker_to_id)
arcface_loader  = DataLoader(arcface_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)

# Pair-based evaluation loaders
train_eval_dataset = SpeakerPairDataset(train_df, mel_transform, amplitude_to_db)
train_eval_loader  = DataLoader(train_eval_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

test_pair_dataset  = SpeakerPairDataset(test_df, mel_transform, amplitude_to_db)
test_pair_loader   = DataLoader(test_pair_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f"ArcFace training: {len(arcface_dataset):,} samples | {len(arcface_loader):,} batches")
print(f"Train eval pairs: {len(train_eval_dataset):,}")
print(f"Test eval pairs:  {len(test_pair_dataset):,}")

## 5. Model Architecture — ResNet-50 Backbone + ArcFace Head

The model has two parts:
1. **ResNet-50 Backbone** → outputs 256-dim L2-normalized embeddings
2. **ArcFace Head** → takes these embeddings and adds the angular margin for classification

During **inference**, we only use the backbone (embeddings).

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import math

class ResNetSpeaker(nn.Module):
    """ResNet-50 backbone that outputs L2-normalized embeddings."""
    def __init__(self, embedding_dim=256):
        super().__init__()
        self.backbone = models.resnet50(weights=None)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.backbone.fc.in_features  # 2048 for ResNet-50
        self.backbone.fc = nn.Identity()
        self.embedding = nn.Linear(in_features, embedding_dim)

    def forward(self, x):
        features  = self.backbone(x)
        embedding = self.embedding(features)
        return F.normalize(embedding, p=2, dim=1)


class ArcFaceHead(nn.Module):
    """
    Additive Angular Margin Loss (ArcFace).
    
    Given L2-normalized embeddings (x) and L2-normalized class weights (W),
    logit = cos(theta), then we add an angular margin 'm' to the angle
    of the true class, making classification harder and forcing tighter clusters.
    
    Args:
        embedding_dim: dimension of input embeddings
        num_classes:   number of speakers
        s:             scale factor (default: 30.0)
        m:             angular margin in radians (default: 0.5)
    """
    def __init__(self, embedding_dim, num_classes, s=30.0, m=0.5):
        super().__init__()
        self.s = s
        self.m = m
        self.W = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.W)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        # Threshold to avoid instability when cos(theta) is very negative
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels):
        # Normalize weights
        W_norm = F.normalize(self.W, p=2, dim=1)

        # cos(theta) = dot product of normalized embeddings and weights
        cosine = F.linear(embeddings, W_norm)  # (batch, num_classes)
        sine   = torch.sqrt(1.0 - torch.pow(cosine, 2)).clamp(min=1e-8)

        # cos(theta + m) = cos(theta)*cos(m) - sin(theta)*sin(m)
        cos_theta_m = cosine * self.cos_m - sine * self.sin_m

        # Safety: when theta > pi - m, use a linear fallback
        cos_theta_m = torch.where(cosine > self.th, cos_theta_m, cosine - self.mm)

        # One-hot encode labels
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)

        # Apply margin only to the true class
        logits = torch.where(one_hot == 1, cos_theta_m, cosine)
        logits = logits * self.s  # scale

        return logits


print(f"ArcFace Head: {NUM_SPEAKERS} classes, scale=30, margin=0.5")

In [ ]:
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model       = ResNetSpeaker(embedding_dim=256).to(device)
arcface     = ArcFaceHead(embedding_dim=256, num_classes=NUM_SPEAKERS, s=30.0, m=0.5).to(device)
criterion   = nn.CrossEntropyLoss()

# Optimize both backbone and ArcFace head together
optimizer   = torch.optim.Adam(
    list(model.parameters()) + list(arcface.parameters()),
    lr=1e-3
)

print("Device:", device)
print("Model:  ResNet-50 (backbone) + ArcFace (head)")
print("Loss:   CrossEntropyLoss (ArcFace logits)")
print("Optim:  Adam (lr=1e-3, joint backbone + head)")

## 6. Training Loop — 25 Epochs (Classification)

- **Classification accuracy** = correctly predicting the speaker ID
- Auto-saves checkpoint after every epoch
- Note: This trains as a **multi-class classification** problem, not pair-based

In [ ]:
from tqdm import tqdm

NUM_EPOCHS  = 25
PRINT_EVERY = 50

loss_history      = []
cls_acc_history   = []  # classification accuracy per epoch

for epoch in range(NUM_EPOCHS):
    model.train()
    arcface.train()
    total_loss    = 0
    total_correct = 0
    total_samples = 0

    bar = tqdm(enumerate(arcface_loader), total=len(arcface_loader),
               desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for batch_idx, (mel, labels) in bar:
        mel    = mel.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        embeddings = model(mel)                   # (batch, 256)
        logits     = arcface(embeddings, labels)  # (batch, num_speakers)

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

        total_loss += loss.item()
        bar.set_postfix(loss=f"{loss.item():.4f}")

        if batch_idx % PRINT_EVERY == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(arcface_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(arcface_loader)
    cls_acc  = total_correct / total_samples

    loss_history.append(avg_loss)
    cls_acc_history.append(cls_acc)

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}] — Loss: {avg_loss:.4f} | Classification Acc: {cls_acc:.4f} ({cls_acc*100:.2f}%)\n")

    torch.save({
        "model_state":      model.state_dict(),
        "arcface_state":    arcface.state_dict(),
        "epoch":            epoch,
        "optimizer_state":  optimizer.state_dict(),
        "loss_history":     loss_history,
        "cls_acc_history":  cls_acc_history,
    }, "checkpoint_resnet50_arcface_5s.pth")
    print(f"  Checkpoint saved → epoch {epoch+1}")

print("\nTraining complete!")

In [ ]:
torch.save({
    "model_state":     model.state_dict(),
    "arcface_state":   arcface.state_dict(),
    "epoch":           NUM_EPOCHS - 1,
    "optimizer_state": optimizer.state_dict(),
    "loss_history":    loss_history,
    "cls_acc_history": cls_acc_history,
}, "checkpoint_resnet50_arcface_5s.pth")
print("Final model saved → checkpoint_resnet50_arcface_5s.pth")

## 7. Training History Graphs

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(loss_history) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("ResNet-50 + ArcFace 5s — Training Metrics", fontsize=13, fontweight='bold')

axes[0].plot(epochs_range, loss_history, marker='o', color='tomato', linewidth=2)
axes[0].set_title("ArcFace Classification Loss per Epoch")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, cls_acc_history, marker='o', color='steelblue', linewidth=2)
axes[1].set_title("Speaker Classification Accuracy per Epoch")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_metrics_arcface_5s.png", dpi=150)
plt.show()
print("Saved → training_metrics_arcface_5s.png")

## 8. Evaluate on Both Datasets (Pair-Based)

Even though we trained with ArcFace (classification), we evaluate the same way as before:
extract embeddings → compute cosine similarity → threshold at 0.5.

In [ ]:
def evaluate_pairs(model, loader, device, label_name="Set"):
    model.eval()
    correct = 0; total = 0
    same_sims, diff_sims = [], []
    same_dists, diff_dists = [], []

    with torch.no_grad():
        for mel1, mel2, labels in tqdm(loader, desc=f"Evaluating {label_name}"):
            mel1   = mel1.to(device)
            mel2   = mel2.to(device)
            labels = labels.to(device)

            emb1 = model(mel1)
            emb2 = model(mel2)

            sim   = F.cosine_similarity(emb1, emb2)
            preds = (sim > 0.5).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            for s, lbl in zip(sim.cpu().tolist(), labels.cpu().tolist()):
                (same_sims if lbl == 1 else diff_sims).append(s)

            d = F.pairwise_distance(emb1, emb2).cpu().tolist()
            for dist, lbl in zip(d, labels.cpu().tolist()):
                (same_dists if lbl == 1 else diff_dists).append(dist)

    acc = correct / total
    print(f"{label_name} Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    return acc, same_sims, diff_sims, same_dists, diff_dists

train_acc, tr_same_sim, tr_diff_sim, tr_same_d, tr_diff_d = evaluate_pairs(
    model, train_eval_loader, device, "Training Set")
test_acc, te_same_sim, te_diff_sim, te_same_d, te_diff_d = evaluate_pairs(
    model, test_pair_loader, device, "Test Set")

print(f"\nTrain Acc: {train_acc*100:.2f}%  |  Test Acc: {test_acc*100:.2f}%")
print(f"Generalisation Gap: {(train_acc - test_acc)*100:.2f}%")

## 9. Accuracy Comparison Bar Chart

In [ ]:
plt.figure(figsize=(7, 5))
bars = plt.bar(["Training Accuracy", "Test Accuracy"],
               [train_acc, test_acc],
               color=["steelblue", "darkorange"], width=0.4, edgecolor='white')
for bar, val in zip(bars, [train_acc, test_acc]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{val*100:.2f}%", ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.ylim(0, 1.1)
plt.ylabel("Accuracy")
plt.title("ResNet-50 + ArcFace 5s — Training vs Test Accuracy", fontsize=12, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("accuracy_comparison_arcface_5s.png", dpi=150)
plt.show()
print("Saved → accuracy_comparison_arcface_5s.png")

## 10. Cosine Similarity Distribution — Training vs Test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Cosine Similarity Distribution (Train vs Test)", fontsize=13, fontweight='bold')

for ax, same, diff, title in [
    (axes[0], tr_same_sim[:5000], tr_diff_sim[:5000], "Training Set"),
    (axes[1], te_same_sim[:5000], te_diff_sim[:5000], "Test Set"),
]:
    ax.hist(same, bins=60, alpha=0.6, color='steelblue', label='Same Speaker (label=1)')
    ax.hist(diff, bins=60, alpha=0.6, color='tomato',    label='Different Speaker (label=0)')
    ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1.5, label='Decision boundary (0.5)')
    ax.set_title(title)
    ax.set_xlabel("Cosine Similarity")
    ax.set_ylabel("Count")
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("cosine_sim_distribution_arcface_5s.png", dpi=150)
plt.show()
print("Saved → cosine_sim_distribution_arcface_5s.png")

## 11. Euclidean Distance Distribution — Training vs Test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Euclidean Distance Distribution (Train vs Test)", fontsize=13, fontweight='bold')

for ax, same, diff, title in [
    (axes[0], tr_same_d[:5000], tr_diff_d[:5000], "Training Set"),
    (axes[1], te_same_d[:5000], te_diff_d[:5000], "Test Set"),
]:
    ax.hist(same, bins=60, alpha=0.6, color='steelblue', label='Same Speaker (label=1)')
    ax.hist(diff, bins=60, alpha=0.6, color='tomato',    label='Different Speaker (label=0)')
    ax.set_title(title)
    ax.set_xlabel("Euclidean Distance")
    ax.set_ylabel("Count")
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("euclidean_dist_arcface_5s.png", dpi=150)
plt.show()
print("Saved → euclidean_dist_arcface_5s.png")

## 12. t-SNE Visualization of Speaker Embeddings

Visualize how well ArcFace has separated different speakers in the embedding space. Each color = one speaker.

In [ ]:
from sklearn.manifold import TSNE

# Extract embeddings from training set (first 2000 samples)
model.eval()
embeddings_list = []
labels_list = []

with torch.no_grad():
    for i, (mel, label) in enumerate(arcface_loader):
        mel = mel.to(device)
        emb = model(mel)
        embeddings_list.append(emb.cpu())
        labels_list.append(label)
        if i >= 30:  # ~2000 samples
            break

all_emb = torch.cat(embeddings_list, dim=0).numpy()
all_lbl = torch.cat(labels_list, dim=0).numpy()

print(f"Running t-SNE on {len(all_emb)} embeddings...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
emb_2d = tsne.fit_transform(all_emb)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=all_lbl, cmap='tab20', s=8, alpha=0.7)
plt.title("t-SNE of Speaker Embeddings (ArcFace)", fontsize=13, fontweight='bold')
plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("tsne_arcface_5s.png", dpi=150)
plt.show()
print("Saved → tsne_arcface_5s.png")

## 13. Final Summary

In [ ]:
print("=" * 55)
print("  MODEL:  ResNet-50 + ArcFace (Angular Margin Loss)")
print("  AUDIO:  5-second clips (80000 samples)")
print("  ARCFACE: s=30, m=0.5 (additive angular margin)")
print("-" * 55)
print(f"  Speaker Classification Acc: {cls_acc_history[-1]*100:.2f}%")
print(f"  Pair-Based Train Accuracy : {train_acc*100:.2f}%")
print(f"  Pair-Based Test Accuracy  : {test_acc*100:.2f}%")
print(f"  Generalisation Gap        : {(train_acc - test_acc)*100:.2f}%")
print("=" * 55)
print("Saved files:")
print("  checkpoint_resnet50_arcface_5s.pth")
print("  training_metrics_arcface_5s.png")
print("  accuracy_comparison_arcface_5s.png")
print("  cosine_sim_distribution_arcface_5s.png")
print("  euclidean_dist_arcface_5s.png")
print("  tsne_arcface_5s.png")